In [6]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. INPUT FILE
# ============================================================
BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data" / "2003_2004" / "Questionnnair"
access_file = DATA_DIR / "HUQ_C.xpt"

xpt_file = Path(access_file)

csv_file = xpt_file.with_name(
    xpt_file.stem + "_corrected.csv"
)


# ============================================================
# 2. READ XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

print("=" * 70)
print("NHANES XPT CHECK")
print("=" * 70)

print("File:", xpt_file.name)
print("Rows:", f"{len(df):,}")
print("Columns:", f"{len(df.columns):,}")


# ============================================================
# 3. NUMERIC COLUMNS
# ============================================================

numeric_cols = df.select_dtypes(include=[np.number]).columns

print("Numeric columns:", len(numeric_cols))


# ============================================================
# 4. CHECK BEFORE CORRECTION
# ============================================================

TINY_VALUE = 5.397605346934028e-79

true_zero_count = int(
    df[numeric_cols].eq(0).sum().sum()
)

tiny_counts = df[numeric_cols].eq(TINY_VALUE).sum()

total_tiny = int(tiny_counts.sum())

tiny_columns = int(
    (tiny_counts > 0).sum()
)


print("\n--- BEFORE CORRECTION ---")

print(
    f"True numeric zeros: {true_zero_count:,}"
)

print(
    f"Tiny value {TINY_VALUE:.16e}: "
    f"{total_tiny:,} occurrences "
    f"across {tiny_columns:,} columns"
)


# ============================================================
# 5. HUQ050 BEFORE
# ============================================================

if "HUQ050" in df.columns:

    print("\n--- HUQ050 BEFORE ---")

    print(
        "True zeros:",
        int(df["HUQ050"].eq(0).sum())
    )

    print(
        "Tiny values:",
        int(df["HUQ050"].eq(TINY_VALUE).sum())
    )

    print("\nValue distribution:")

    print(
        df["HUQ050"]
        .value_counts(dropna=False)
        .sort_index()
    )


# ============================================================
# 6. RESTORE TRUE ZEROS
# ============================================================

for col in numeric_cols:

    mask = df[col].eq(TINY_VALUE)

    if mask.any():
        df.loc[mask, col] = 0


# ============================================================
# 7. CHECK AFTER CORRECTION
# ============================================================

numeric_df = df[numeric_cols]

true_zero_count_after = int(
    numeric_df.eq(0).sum().sum()
)

tiny_count_after = int(
    numeric_df.eq(TINY_VALUE).sum().sum()
)


print("\n--- AFTER CORRECTION ---")

print(
    f"True numeric zeros: "
    f"{true_zero_count_after:,}"
)

print(
    f"Remaining tiny values: "
    f"{tiny_count_after:,}"
)


# ============================================================
# 8. HUQ050 AFTER
# ============================================================

if "HUQ050" in df.columns:

    print("\n--- HUQ050 AFTER ---")

    print(
        "True zeros:",
        int(df["HUQ050"].eq(0).sum())
    )

    print(
        "Tiny values:",
        int(df["HUQ050"].eq(TINY_VALUE).sum())
    )

    print("\nValue distribution:")

    print(
        df["HUQ050"]
        .value_counts(dropna=False)
        .sort_index()
    )


# ============================================================
# 9. DESCRIPTIVE STATISTICS
# ============================================================

print("\n--- DESCRIPTIVE STATISTICS ---")

stats = numeric_df.describe().T

stats["missing"] = numeric_df.isna().sum()

stats["zero_count"] = numeric_df.eq(0).sum()

stats["unique_values"] = numeric_df.nunique()

print(stats)


# ============================================================
# 10. EXPORT CSV
# ============================================================

df.to_csv(
    csv_file,
    index=False
)

print("\nCSV created:")
print(csv_file)


# ============================================================
# 11. FINAL VALIDATION
# ============================================================

remaining_tiny = int(
    df.select_dtypes(include=[np.number])
      .eq(TINY_VALUE)
      .sum()
      .sum()
)

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

if remaining_tiny == 0:
    print("PASS: No tiny zero-conversion values remain.")
else:
    print(
        f"FAIL: {remaining_tiny:,} tiny values remain."
    )

if "HUQ050" in df.columns:

    huq_zeros = int(
        df["HUQ050"].eq(0).sum()
    )

    print(
        f"HUQ050 true zero count: {huq_zeros:,}"
    )

NHANES XPT CHECK
File: HUQ_C.xpt
Rows: 10,122
Columns: 16
Numeric columns: 16

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny value 5.3976053469340279e-79: 1,251 occurrences across 2 columns

--- HUQ050 BEFORE ---
True zeros: 0
Tiny values: 1250

Value distribution:
HUQ050
5.397605e-79    1250
1.000000e+00    2082
2.000000e+00    2987
3.000000e+00    2551
4.000000e+00     615
5.000000e+00     630
9.900000e+01       7
Name: count, dtype: int64

--- AFTER CORRECTION ---
True numeric zeros: 1,251
Remaining tiny values: 0

--- HUQ050 AFTER ---
True zeros: 1250
Tiny values: 0

Value distribution:
HUQ050
0.0     1250
1.0     2082
2.0     2987
3.0     2551
4.0      615
5.0      630
99.0       7
Name: count, dtype: int64

--- DESCRIPTIVE STATISTICS ---
           count          mean          std      min       25%      50%  \
SEQN     10122.0  26065.500000  2922.114046  21005.0  23535.25  26065.5   
HUQ010   10122.0      2.284331     1.130708      1.0      1.00      2.0   
HUQ020    9645